In [ ]:
import re
import numpy as np
import pandas as pd

def pretty(df: pd.DataFrame, title: str = ""):
    display(
        df.style.set_properties(
            **{
                "text-align": "center",
                "white-space": "nowrap",
                "font-family": "monospace",
                "padding": "4px",
            }
        )
    )


def print_xai_metrics(csv_path: str, B: int = 5):
    df = pd.read_csv(csv_path).copy()
    key_path = "path"
    key_pix = "pixel_count"

    global_rows = df[df["roi_type"] == "global"][[key_path, "class_id", key_pix]].drop_duplicates()
    global_rows["_has_global"] = True

    df = df.merge(global_rows, on=[key_path, "class_id", key_pix], how="left")
    mask_local_to_drop = (df["roi_type"] == "local") & df["_has_global"].notna()
    df = df[~mask_local_to_drop].copy()
    df = df.drop(columns=["_has_global"])

    attr_cols = []
    for c in df.columns:
        m = re.fullmatch(r"attr_c(\d+)", c)
        if m is not None:
            idx = int(m.group(1))
            attr_cols.append((idx, c))
    attr_cols = [c for _, c in sorted(attr_cols, key=lambda x: x[0])]

    numeric_cols = [
        "faithfulness_r_raw",
        "sensitivity",
        "complexity",
    ]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["faithfulness"] = df["faithfulness_r_raw"]

    t = pd.to_numeric(df["time_ms_attr"], errors="coerce")
    runs = (
        t.isna()
        | (t != t.shift())
        | (df["roi_type"] != df["roi_type"].shift())
        | (df["method"] != df["method"].shift())
    ).cumsum()
    run_len = runs.map(runs.value_counts())
    df["time_ms_row"] = t / run_len

    gbl = df[df["roi_type"] == "global"].copy()
    loc = df[df["roi_type"] == "local"].copy()

    keys = ["image_id", "class_id", "method"]
    g_map = gbl[keys + attr_cols].rename(columns={c: f"{c}_g" for c in attr_cols})

    g_pix = (
        gbl[keys + [key_pix]]
        .drop_duplicates(subset=keys)
        .rename(columns={key_pix: f"{key_pix}_g"})
    )

    loc = loc.copy()
    loc["__idx__"] = loc.index
    loc = loc.merge(g_pix, on=keys, how="left")
    loc["pixel_ratio"] = pd.to_numeric(loc[key_pix], errors="coerce") / pd.to_numeric(
        loc[f"{key_pix}_g"], errors="coerce"
    )

    m = loc.merge(g_map, on=keys, how="left")

    A = m[attr_cols].to_numpy(float)
    B_vec = m[[f"{c}_g" for c in attr_cols]].to_numpy(float)

    denom = np.linalg.norm(A, axis=1) * np.linalg.norm(B_vec, axis=1)
    cos = (A * B_vec).sum(axis=1) / np.where(denom == 0, np.nan, denom)

    m["cos_sim"] = cos

    df["cos_sim"] = np.nan
    df.loc[m["__idx__"], "cos_sim"] = m["cos_sim"].to_numpy()

    agg_cos = (
        m.groupby(keys)
        .agg(cos_sim_mean=("cos_sim", "mean"))
        .reset_index()
    )
    df = df.merge(agg_cos, on=keys, how="left")
    df.loc[df["roi_type"] == "global", "cos_sim"] = df["cos_sim_mean"]

    def fmt_mean_sd(arr: pd.Series):
        x = pd.to_numeric(arr, errors="coerce").to_numpy(dtype=float)
        x = x[np.isfinite(x)]
        if x.size == 0:
            return "nan"
        mu = float(np.mean(x))
        sd = float(np.std(x, ddof=1)) if x.size >= 2 else 0.0
        return f"{mu:.4f} ± {sd:.4f}"

    def summarize_methods_only(d, by_cols):
        rows = []
        for key_vals, d_group in d.groupby(by_cols, dropna=False):
            if not isinstance(key_vals, tuple):
                key_vals = (key_vals,)

            row = {}
            for col, val in zip(by_cols, key_vals):
                row[col] = val

            subset = ["roi_type", "class_id", key_path, "method"]
            if str(d_group["roi_type"].iloc[0]) == "local":
                subset = subset + ["component_idx"]
            row["n"] = int(d_group.drop_duplicates(subset=subset).shape[0])

            row["faithfulness"] = fmt_mean_sd(d_group["faithfulness"])
            row["sensitivity"] = fmt_mean_sd(d_group["sensitivity"])
            row["complexity"] = fmt_mean_sd(d_group["complexity"])
            row["time"] = fmt_mean_sd(d_group["time_ms_row"])
            row["cos_sim"] = fmt_mean_sd(d_group["cos_sim"])

            rows.append(row)

        cols = list(by_cols) + [
            "n",
            "faithfulness",
            "sensitivity",
            "complexity",
            "time",
            "cos_sim",
        ]
        result = pd.DataFrame(rows)
        return result[cols]

    res = summarize_methods_only(df, ["roi_type", "method"])
    pretty(res, "PER METHOD (global vs local)")

    m_small = m[(m["pixel_ratio"] <= 0.5) & np.isfinite(m["cos_sim"])].copy()

    pct_rows = []
    for class_id, g in m_small.groupby("class_id", dropna=False):
        s = g["cos_sim"].to_numpy(dtype=float)
        p05, p50, p95 = np.nanpercentile(s, [5, 50, 95]) if s.size else (np.nan, np.nan, np.nan)
        pct_rows.append(
            {
                "class_id": class_id,
                "cos_sim_p05": float(p05),
                "cos_sim_p50": float(p50),
                "cos_sim_p95": float(p95),
                "n": int(np.isfinite(s).sum()),
            }
        )

    summary_pct = pd.DataFrame(pct_rows).sort_values("class_id").reset_index(drop=True)
    pretty(summary_pct, "COS_SIM PERCENTILES PER CLASS (all methods, local pixel_ratio<=0.5)")


In [ ]:
print_xai_metrics("/df_xai/xai_clousden12.csv")

/tmp/ipython-input-2961303183.py:20: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path).copy()


  dataset_name  image_id                                               path  \
0     cloudsen         1  /content/drive/MyDrive/data/cloudsen12_data/cl...   
1     cloudsen         2  /content/drive/MyDrive/data/cloudsen12_data/cl...   
2     cloudsen         3  /content/drive/MyDrive/data/cloudsen12_data/cl...   
3     cloudsen         4  /content/drive/MyDrive/data/cloudsen12_data/cl...   
4     cloudsen         5  /content/drive/MyDrive/data/cloudsen12_data/cl...   

  roi_type  class_id  component_idx  pixel_count  connectivity    H    W  ...  \
0   global         0            NaN        62656             4  512  512  ...   
1   global         0            NaN        16466             4  512  512  ...   
2   global         0            NaN       224146             4  512  512  ...   
3   global         0            NaN       184664             4  512  512  ...   
4   global         0            NaN       262144             4  512  512  ...   

   attr_c10  attr_c11  attr_c12  use_a

,roi_type,method,n,faithfulness,sensitivity,complexity,time,cos_sim
0,global,ig_0,1392,0.0831 ± 0.3227,0.3677 ± 0.3405,0.7528 ± 0.0803,92.0704 ± 0.7025,0.9356 ± 0.1406
1,global,ig_mean,1392,0.0595 ± 0.3359,0.4305 ± 0.3495,0.7581 ± 0.0813,91.6980 ± 0.6807,0.9341 ± 0.1301
2,global,lime_0,1392,0.1194 ± 0.2998,0.4386 ± 0.1932,0.8256 ± 0.0633,227.8787 ± 4.2373,0.9282 ± 0.0553
3,global,lime_mean,1392,0.1603 ± 0.2828,0.4809 ± 0.2986,0.8082 ± 0.0851,224.9714 ± 22.0046,0.9045 ± 0.1115
4,global,lime_noise,1392,0.6932 ± 0.2587,0.5219 ± 0.3780,0.7624 ± 0.0613,216.8815 ± 22.6551,0.9483 ± 0.0771
5,global,reg_occ_0,1392,0.1876 ± 0.3064,1.1556 ± 0.4313,0.8305 ± 0.0604,57.8492 ± 5.9360,0.9689 ± 0.0540
6,global,reg_occ_mean,1392,0.2831 ± 0.3190,0.9349 ± 0.2331,0.7961 ± 0.0817,58.7691 ± 5.4386,0.9629 ± 0.0778
7,global,reg_occ_noise,1392,0.7069 ± 0.2241,0.9178 ± 0.3861,0.8033 ± 0.0688,163.1419 ± 8.7425,0.9706 ± 0.0714
8,global,smoothgrad,1392,0.2800 ± 0.3503,0.0456 ± 0.0276,0.9335 ± 0.0068,56.4983 ± 0.5265,0.9996 ± 0.0007
9,global,vanilla_grad,1392,0.3099 ± 0.3612,0.1147 ± 0.0580,0.9176 ± 0.0127,5.8193 ± 1.1354,0.9990 ± 0.0020


,class_id,cos_sim_p05,cos_sim_p50,cos_sim_p95,n
0,0,0.416361,0.939088,0.999822,2820
1,1,0.930244,0.992668,0.999854,7040
2,2,0.185734,0.948833,0.999794,940
3,3,0.775908,0.970868,0.999830,7070


In [ ]:
print_xai_metrics("df_xai/xai_cloudsen12plus.csv")

/tmp/ipython-input-3438927238.py:20: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path).copy()


,roi_type,method,n,faithfulness,sensitivity,complexity,time,cos_sim
0,global,ig_0,1434,0.0589 ± 0.2981,0.4593 ± 0.3893,0.7675 ± 0.0773,91.5561 ± 0.3028,0.9385 ± 0.1301
1,global,ig_mean,1434,0.0491 ± 0.3280,0.5535 ± 0.4065,0.7763 ± 0.0796,91.1941 ± 0.3056,0.9224 ± 0.1431
2,global,lime_0,1434,0.2239 ± 0.2518,0.4977 ± 0.2085,0.8318 ± 0.0589,228.0481 ± 3.5545,0.9081 ± 0.0779
3,global,lime_mean,1434,0.2198 ± 0.2817,0.5202 ± 0.3149,0.8087 ± 0.0836,227.2244 ± 23.4117,0.8999 ± 0.1083
4,global,lime_noise,1434,0.6966 ± 0.2597,0.5158 ± 0.3498,0.7609 ± 0.0714,216.4400 ± 22.7834,0.9397 ± 0.0871
5,global,reg_occ_0,1434,0.2713 ± 0.2586,1.0738 ± 0.4075,0.8260 ± 0.0609,58.4074 ± 6.3633,0.9667 ± 0.0665
6,global,reg_occ_mean,1434,0.3412 ± 0.3000,0.9220 ± 0.2126,0.7775 ± 0.0763,59.4558 ± 4.2379,0.9670 ± 0.0693
7,global,reg_occ_noise,1434,0.6959 ± 0.2385,0.8705 ± 0.3549,0.7884 ± 0.0793,164.8664 ± 4.5724,0.9670 ± 0.0795
8,global,smoothgrad,1434,0.3489 ± 0.2808,0.0404 ± 0.0253,0.9318 ± 0.0093,56.3149 ± 0.7294,0.9995 ± 0.0024
9,global,vanilla_grad,1434,0.3761 ± 0.3009,0.1059 ± 0.0588,0.9217 ± 0.0113,5.6254 ± 0.9145,0.9991 ± 0.0033


,class_id,cos_sim_p05,cos_sim_p50,cos_sim_p95,n
0,0,0.336450,0.916353,0.999687,1150
1,1,0.916210,0.990889,0.999816,7150
2,2,0.132698,0.932345,0.999780,1210
3,3,0.678349,0.965443,0.999821,7150
